# Modelo preditivo — NPS (opcional do desafio)

**Pergunta:** com dados operacionais disponíveis **antes** da pesquisa NPS, estimar a nota (regressão) e identificar pedidos em risco.

- **Alvo:** `nps_score` (regressão). Complemento: métricas de classificação binária **detrator** (nota < 7) para decisão operacional.
- **Entradas:** pedido, logística, atendimento (sem vazamento de informação futura). Excluímos `customer_id` e `order_id` do modelo.
- **Separação:** 80% treino / 20% teste, `random_state=42` (base transversal; em produção, validar por tempo).
- **Modelo:** `HistGradientBoostingRegressor` (bom padrão em dados tabulares heterogêneos).
- **Avaliação:** MAE, RMSE, R² no teste; importância de features.
- **Uso prático:** priorizar inspeção de pedidos com **NPS predito baixo** antes do envio da pesquisa; acionar logística/atendimento.

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

_cwd = Path.cwd().resolve()
PROJECT_ROOT = _cwd if (_cwd / "data" / "processed" / "dataset_processado.csv").exists() else _cwd.parent
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "dataset_processado.csv"
MODELS = PROJECT_ROOT / "models"
REPORTS = PROJECT_ROOT / "reports"
MODELS.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)

feature_cols = [
    "customer_age",
    "customer_region",
    "customer_tenure_months",
    "order_value",
    "items_quantity",
    "discount_value",
    "payment_installments",
    "delivery_time_days",
    "delivery_delay_days",
    "freight_value",
    "delivery_attempts",
    "customer_service_contacts",
    "resolution_time_days",
    "complaints_count",
    "repeat_purchase_30d",
    "csat_internal_score",
]
target = "nps_score"

X = df[feature_cols]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

categorical = ["customer_region"]
numeric = [c for c in feature_cols if c not in categorical]

preprocess = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
    ]
)

model = HistGradientBoostingRegressor(max_depth=6, learning_rate=0.06, max_iter=300, random_state=42)
pipe = Pipeline([("prep", preprocess), ("model", model)])
pipe.fit(X_train, y_train)

pred = pipe.predict(X_test)
mae = mean_absolute_error(y_test, pred)
rmse = root_mean_squared_error(y_test, pred)
r2 = r2_score(y_test, pred)
print(f"MAE: {mae:.3f} | RMSE: {rmse:.3f} | R2: {r2:.3f}")

# Classificação derivada: detrator < 7
y_bin = (y_test < 7).astype(int)
pred_bin = (pred < 7).astype(int)
acc = (y_bin == pred_bin).mean()
print(f"Acurácia detrator<7 (proxy): {acc:.3f}")

joblib.dump(pipe, MODELS / "pipeline_nps_hgb.joblib")
print("Modelo salvo em models/pipeline_nps_hgb.joblib")

In [ ]:
# Importância por permutação (no conjunto de teste)
r = permutation_importance(pipe, X_test, y_test, n_repeats=15, random_state=42, n_jobs=-1)
# importances_mean tem o mesmo comprimento que colunas de X (pré-transformação)
imp = pd.Series(r.importances_mean, index=X_test.columns).sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(8, 5))
imp.sort_values().plot.barh(ax=ax, color="#264653")
ax.set_title("Top features — importância por permutação")
plt.tight_layout()
fig.savefig(REPORTS / "fig09_importancia_modelo.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Predito vs real (teste)
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(y_test, pred, alpha=0.3, s=12)
lims = 0, 10
ax.plot(lims, lims, "r--", lw=1)
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel("NPS real")
ax.set_ylabel("NPS predito")
ax.set_title("Teste: predito vs real")
plt.tight_layout()
fig.savefig(REPORTS / "fig10_pred_vs_real.png", dpi=150, bbox_inches="tight")
plt.show()